<a href="https://colab.research.google.com/github/lmassaron/fine-tuning-workshop/blob/main/06_medical_synthetic_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Workshop: Tunix-Med · Part 1: Building the Medical Mind

In this notebook, we automate the creation of a high-quality medical dataset focused on **Cardiology**. Domain adaptation requires data that is both **factually dense** and **properly formatted**.

### The Synthetic Data Strategy
Real-world clinical data is often private (HIPAA/GDPR). Synthetic data allows us to:
1.  **Extract Knowledge**: Convert unstructured text (Wikipedia) into structured Q&A.
2.  **Control Quality**: Use a "Teacher LLM" to generate and a "Judge LLM" to score the data.
3.  **Scale**: Generate thousands of high-quality examples in hours using **vLLM**.

### Pipeline Overview
1.  **Wikipedia Scraping**: Gathering raw cardiology text.
2.  **vLLM Acceleration**: Running a fast inference server to handle high-volume generation.
3.  **Create & Curate**: Using `synthetic-data-kit` to generate QA pairs and filter them based on a quality threshold (e.g., only keeping pairs with a score > 7).
4.  **Dataset Export**: Pushing the final dataset to the Hugging Face Hub.

In [1]:
import os
import re
import time
import subprocess
import wikipediaapi
import pandas as pd
from tqdm import tqdm
from datasets import Dataset, DatasetDict

# --- Workshop Settings ---
DEMO = False  # Set to True for a quick 5-minute run
MODEL_ENGINE = "Qwen/Qwen2.5-7B-Instruct-AWQ"
NUM_PAIRS_PER_FILE = 30
QUALITY_THRESHOLD = 7.0


def init():
    """Scaffold the local data directory structure."""
    for d in ["medical_input", "generated", "curated", "final"]:
        os.makedirs(f"data/{d}", exist_ok=True)


init()

## 1 · Domain Scraping

We use the `wikipedia-api` to fetch content for a targeted list of 70+ cardiology topics. We clean the text to remove Wikipedia-specific artifacts (like citations) to ensure the LLM receives clean context.

In [2]:
def clean_string(input_string):
    """Remove whitespace and bracketed citations."""
    cleaned_string = " ".join(input_string.split())
    cleaned_string = re.sub(r"\{.*?\}|\}", "", cleaned_string)
    return cleaned_string


def get_wikipedia_content(titles):
    """Crawl Wikipedia sections for relevant medical knowledge."""
    wiki_wiki = wikipediaapi.Wikipedia("MedicalBot/1.0", "en")
    extracted_texts = []
    for title in tqdm(titles):
        page = wiki_wiki.page(title)
        if page.exists():
            # Add the summary
            extracted_texts.append(f"{page.title} : {clean_string(page.summary)}")
            # Add non-trivial sections
            for section in page.sections:
                if len(section.text) > 200:
                    extracted_texts.append(
                        f"{page.title} ({section.title}) : {clean_string(section.text)}"
                    )
    return extracted_texts


# Extensive list of cardiology topics to build a comprehensive knowledge base
cardiology_topics = [
    "Hypertension",
    "Myocardial infarction",
    "Heart failure",
    "Atrial fibrillation",
    "Brugada syndrome",
    "Echocardiography",
    "Statin",
    "Beta blocker",
    "ACE inhibitor",
]  # ... (full list truncated for brevity)

if DEMO:
    cardiology_topics = cardiology_topics[:3]

extracted_texts = get_wikipedia_content(cardiology_topics)
for i, text in enumerate(extracted_texts):
    with open(f"data/medical_input/med_{i}.txt", "w") as f:
        f.write(text)

100%|██████████| 9/9 [00:05<00:00,  1.50it/s]


## 2 · vLLM Acceleration

Generating 10,000+ QA pairs would take days with standard library inference. We use **vLLM** (Variable Large Language Model) which uses **PagedAttention** to serve models at extreme speeds. We run it as a background process.

In [3]:
def start_vllm():
    """Start the vLLM OpenAI-compatible server in the background."""
    vllm_log = open("vllm_server.log", "w")
    return subprocess.Popen(
        [
            "vllm",
            "serve",
            MODEL_ENGINE,
            "--port",
            "8000",
            "--enforce-eager",
            "--gpu-memory-utilization",
            "0.25",
            "--max-model-len",
            "8192",
            "--disable-log-stats",
            "--uvicorn-log-level",
            "error",
        ],
        stdout=vllm_log,
        stderr=vllm_log,
    )


vllm_proc = start_vllm()
print("Waiting for vLLM server to be ready...")
while "Starting vLLM server" not in open("vllm_server.log").read():
    time.sleep(2)
print("vLLM server ready!")

Waiting for vLLM server to be ready...


KeyboardInterrupt: 

## 3 · Create & Curate Loop

We use `synthetic-data-kit` to automate the heavy lifting. The config below defines the medical system prompts and the grading rubric for the Judge LLM.

In [ ]:
# Generate the YAML config for the synthetic-data-kit
yaml_content = f"""
paths:
  input: {{ txt: "data/medical_input" }}
  output: {{ generated: "data/generated", curated: "data/curated", final: "data/final" }}
vllm:
  api_base: "http://localhost:8000/v1"
  model: "{MODEL_ENGINE}"
generation:
  num_pairs: {NUM_PAIRS_PER_FILE}
curate:
  threshold: {QUALITY_THRESHOLD}
prompts:
  qa_generation: |
    Create question-answer pairs from this medical text. Focus on diagnostic criteria and treatments.
    Text: {{{{text}}}}
  qa_rating: |
    Rate these QA pairs from 1-10 based on accuracy and clinical utility.
    {{{{pairs}}}}
"""
with open("medical_data_config.yaml", "w") as f:
    f.write(yaml_content)

input_files = [
    os.path.join("data/medical_input", f)
    for f in os.listdir("data/medical_input")
    if f.endswith(".txt")
]
for filepath in tqdm(input_files):
    # Step 1: Create raw pairs
    subprocess.run(
        [
            "synthetic-data-kit",
            "-c",
            "medical_data_config.yaml",
            "create",
            filepath,
            "--type",
            "qa",
        ],
        check=True,
    )

    # Step 2: Curate (Judge) pairs
    gen_file = os.path.join(
        "data/generated", os.path.basename(filepath).replace(".txt", "_qa_pairs.json")
    )
    if os.path.exists(gen_file):
        subprocess.run(
            [
                "synthetic-data-kit",
                "-c",
                "medical_data_config.yaml",
                "curate",
                gen_file,
            ],
            check=True,
        )

    # Step 3: Save in Fine-Tuning (FT) format
    curated_file = os.path.join(
        "data/curated",
        os.path.basename(filepath).replace(".txt", "_qa_pairs_cleaned.json"),
    )
    if os.path.exists(curated_file):
        subprocess.run(
            [
                "synthetic-data-kit",
                "-c",
                "medical_data_config.yaml",
                "save-as",
                curated_file,
                "-f",
                "ft",
            ],
            check=True,
        )

vllm_proc.terminate()
print("Generation and Curation complete!")

## 4 · Final Dataset Formatting

We finalize the dataset by applying the standard **System Prompt** to every conversation. This ensures the model learns to associate the "Knowledgeable medical assistant" persona with accurate clinical output.

In [ ]:
SYSTEM_PROMPT = "You are a knowledgeable medical assistant specializing in cardiology. Answer clinical questions accurately, focusing on diagnostic criteria, treatment guidelines, and pathophysiology."


def set_system_prompt(row):
    row["messages"][0]["content"] = SYSTEM_PROMPT
    return row


# Merge all cleaned files into a single dataframe
conversations = pd.concat(
    [pd.read_json(f"data/final/{n}") for n in os.listdir("data/final")]
).reset_index(drop=True)
conversations = conversations.apply(set_system_prompt, axis=1)

complete_dataset = DatasetDict({"train": Dataset.from_pandas(conversations)})
complete_dataset.save_to_disk("data/medical-cardiology-qa")
print(f"Final dataset ready: {len(conversations)} cardiology clinical pairs.")